# BLE-GSpeed: Data Cleaning, Temporal Aggregation, and Preprocessing

This notebook continues the existing BLE-GSpeed workflow and replaces the outdated synthetic-dataset assumptions (`Reg_Feat_*`, `Cat_Feat_*`, and `Target_Regression`) with the actual columns:

`mac`, `rssi`, `device`, `timestamp`, `user`, `direction`, `walk_id`, `speed`.

**Goal:** prepare a leakage-aware, window-aggregated dataset for regression. This notebook intentionally stops before fitting a regression estimator.

### Workflow
1. Load the data.
2. Validate and clean required fields.
3. Forward-fill missing RSSI within the same user, walking session, and device.
4. Aggregate observations into temporal windows.
5. Select model features and target.
6. Split by walking session to avoid session leakage.
7. Fit preprocessing on training data only.
8. Validate and inspect the processed matrices.

**Important:** `WINDOW_MS = 500` is an initial setting, not a proven optimum. Later, repeat the experiment with 250 ms and 1,000 ms using the same splitting strategy. The timestamp is assumed to be in milliseconds, consistent with the sample values previously inspected.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# -----------------------------
# Configuration
# -----------------------------
DATA_PATH = Path("ble-gspeed.csv")
TARGET = "speed"
WINDOW_MS = 500
RANDOM_STATE = 42
TEST_SIZE = 0.20

REQUIRED_COLUMNS = [
    "mac", "rssi", "device", "timestamp",
    "user", "direction", "walk_id", "speed"
]

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Place the notebook in the same folder as ble-gspeed.csv, "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing from dataset: {missing_columns}")

print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(df.head())

## Step 2 — Validate and clean the raw data

We report duplicate rows and out-of-range RSSI values rather than silently deleting them. Rows missing a target, timestamp, or session identifiers cannot be used reliably for supervised, session-aware aggregation and are removed. Non-numeric or non-finite target values are also excluded. A non-positive speed is flagged and excluded because the known BLE-GSpeed target range is positive; review this rule if the dataset documentation defines valid zero-speed observations.

In [ ]:
# Work on a copy so the original loaded data remains available as `df`
raw_df = df.copy()
df = df.copy()

print("Initial shape:", df.shape)
print("\nMissing values before cleaning:")
display(df.isna().sum().rename("missing_count").to_frame())

# Report exact duplicate rows; do not automatically remove them.
exact_duplicates = int(df.duplicated().sum())
print(f"Exact duplicate rows (reported, not automatically removed): {exact_duplicates:,}")

# Convert columns used in calculations to numeric; invalid strings become NaN.
for col in ["rssi", "timestamp", "user", "direction", "walk_id", "speed"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Replace +/- infinity with NaN in numeric fields.
numeric_validation_cols = ["rssi", "timestamp", "user", "direction", "walk_id", "speed"]
df[numeric_validation_cols] = df[numeric_validation_cols].replace([np.inf, -np.inf], np.nan)

# Required to group chronologically and assign supervised labels.
required_for_rows = ["timestamp", "user", "walk_id", "speed"]
before_required_filter = len(df)
df = df.dropna(subset=required_for_rows).copy()
print(f"Rows removed for missing/invalid timestamp, user, walk_id, or speed: {before_required_filter - len(df):,}")

# Speed should be positive for this dataset. Report and remove non-positive labels.
nonpositive_speed = int((df["speed"] <= 0).sum())
print(f"Rows with speed <= 0 (reported and excluded): {nonpositive_speed:,}")
df = df.loc[df["speed"] > 0].copy()

# RSSI is expected to be numeric. Report unusual values; don't silently clip/remove.
invalid_rssi_range = int((df["rssi"].notna() & ~df["rssi"].between(-127, 0)).sum())
print(f"Non-missing RSSI values outside [-127, 0] dBm (reported only): {invalid_rssi_range:,}")

# Missing identifiers cannot be used to establish a trustworthy session/device group.
for col in ["mac", "device"]:
    df[col] = df[col].astype("string").str.strip()
    df.loc[df[col].isin(["", "nan", "None", "<NA>"]), col] = pd.NA

before_id_filter = len(df)
df = df.dropna(subset=["user", "walk_id", "device"]).copy()
print(f"Rows removed for missing user, walk_id, or device: {before_id_filter - len(df):,}")

# Keep consistent integer-like identifiers where possible.
for col in ["user", "walk_id", "direction"]:
    if df[col].notna().all() and np.allclose(df[col], np.round(df[col])):
        df[col] = df[col].astype("int64")

# A session is uniquely identified by the pair (user, walk_id).
df["session_id"] = (
    df["user"].astype(str) + "__walk_" + df["walk_id"].astype(str)
)

# Check whether a session has one target value. Aggregating a single label per
# window is only valid when the speed label is constant within each session.
session_label_counts = df.groupby("session_id")["speed"].nunique(dropna=True)
sessions_with_multiple_labels = int((session_label_counts > 1).sum())

print(f"\nRows after basic validation: {len(df):,}")
print(f"Distinct sessions: {df['session_id'].nunique():,}")
print(f"Sessions with more than one speed label: {sessions_with_multiple_labels:,}")

if sessions_with_multiple_labels:
    raise ValueError(
        "Some (user, walk_id) sessions have multiple speed labels. "
        "Review the target semantics before assigning one label to each window."
    )

print("\nMissing values after basic validation:")
display(df[REQUIRED_COLUMNS].isna().sum().rename("missing_count").to_frame())

## Step 3 — Impute missing RSSI with the previous value

RSSI is forward-filled within `(user, walk_id, device)` after sorting by timestamp. This prevents a measurement from one device or walking session being copied into another. We do **not** backward-fill, because that would use a future measurement. If a value is still missing after forward-fill (for example, the first observation for a device in a session), it is excluded from aggregation and counted in the output.

In [ ]:
# Sort before forward-fill so "previous" means previous in time.
df = df.sort_values(
    ["user", "walk_id", "device", "timestamp"]
).reset_index(drop=True)

missing_rssi_before = int(df["rssi"].isna().sum())
print("Missing RSSI before forward-fill:", missing_rssi_before)

df["rssi"] = (
    df.groupby(["user", "walk_id", "device"], sort=False)["rssi"]
      .ffill()
)

missing_rssi_after = int(df["rssi"].isna().sum())
print("Missing RSSI after forward-fill:", missing_rssi_after)
print("RSSI values filled:", missing_rssi_before - missing_rssi_after)

# Rows still missing RSSI cannot contribute to RSSI summary features.
# No backward-fill or global-median substitution is performed.
remaining_missing_rows = int(df["rssi"].isna().sum())
if remaining_missing_rows:
    print(
        f"Excluding {remaining_missing_rows:,} rows whose RSSI is still missing "
        "after session/device-aware forward-fill."
    )
    df = df.dropna(subset=["rssi"]).copy()

print("Rows available for aggregation:", f"{len(df):,}")
print("Missing RSSI remaining:", int(df["rssi"].isna().sum()))

## Step 4 — Temporal window aggregation and feature engineering

Windows are anchored to the start of each walking session, so window boundaries do not mix sessions. The initial window size is `WINDOW_MS = 500`. Each row in the aggregated dataset represents one `(session, time window)`.

Features include RSSI mean, median, standard deviation, minimum, maximum, range, observation count, and counts of unique devices and MAC addresses. The most common direction is retained as a categorical feature. Absolute timestamps, window IDs, user IDs, walking IDs, and MAC/device identities are not passed directly as predictors. Session identifiers are retained only for splitting.

Because the known dataset has one speed label per session, each window receives that session's label. This creates more window-level rows, but **does not create more independent walking sessions**; evaluation therefore remains grouped by session.

In [ ]:
# Create elapsed time within each session, then assign non-overlapping windows.
session_start_ms = df.groupby("session_id")["timestamp"].transform("min")
df["elapsed_ms"] = df["timestamp"] - session_start_ms
df["window_id"] = (df["elapsed_ms"] // WINDOW_MS).astype("int64")

def mode_or_nan(values):
    modes = values.mode(dropna=True)
    return modes.iloc[0] if not modes.empty else np.nan

window_df = (
    df.groupby(["session_id", "window_id"], sort=True)
      .agg(
          user=("user", "first"),
          walk_id=("walk_id", "first"),
          window_start_ms=("timestamp", "min"),
          window_end_ms=("timestamp", "max"),
          rssi_mean=("rssi", "mean"),
          rssi_median=("rssi", "median"),
          rssi_std=("rssi", "std"),
          rssi_min=("rssi", "min"),
          rssi_max=("rssi", "max"),
          n_observations=("rssi", "size"),
          n_unique_devices=("device", "nunique"),
          n_unique_macs=("mac", "nunique"),
          direction_mode=("direction", mode_or_nan),
          speed=(TARGET, "first"),
      )
      .reset_index()
)

# A one-observation window has undefined sample standard deviation; use 0
# to indicate no observed variation within that window.
window_df["rssi_std"] = window_df["rssi_std"].fillna(0.0)
window_df["rssi_range"] = window_df["rssi_max"] - window_df["rssi_min"]

# Duration is a diagnostic only, not a model predictor.
window_df["observed_span_ms"] = (
    window_df["window_end_ms"] - window_df["window_start_ms"]
)

print(f"Window size: {WINDOW_MS} ms")
print(f"Raw observations used: {len(df):,}")
print(f"Aggregated window rows: {len(window_df):,}")
print(f"Distinct sessions in aggregated data: {window_df['session_id'].nunique():,}")

print("\nWindow rows per session:")
display(window_df.groupby("session_id").size().describe().to_frame("windows_per_session"))

print("\nAggregated data preview:")
display(window_df.head())

print("\nMissing values in aggregated data:")
display(window_df.isna().sum().rename("missing_count").to_frame())

## Step 5 — Select predictors and target

The initial feature set intentionally excludes `user`, `walk_id`, `session_id`, raw timestamps, and `window_id` from predictors. Those columns can identify a person, session, or point in a session and may encourage memorization rather than learning a general RSSI–speed relationship.

`direction_mode` is treated as categorical. RSSI summary statistics and measurement counts are numeric. Device and MAC identities are represented only through counts of distinct values in each window, not through their raw identifiers.

In [ ]:
TARGET = "speed"

numeric_features = [
    "rssi_mean",
    "rssi_median",
    "rssi_std",
    "rssi_min",
    "rssi_max",
    "rssi_range",
    "n_observations",
    "n_unique_devices",
    "n_unique_macs",
]

categorical_features = ["direction_mode"]
feature_columns = numeric_features + categorical_features

missing_feature_columns = sorted(set(feature_columns + [TARGET, "session_id"]) - set(window_df.columns))
if missing_feature_columns:
    raise ValueError(f"Expected feature/target columns not found: {missing_feature_columns}")

X = window_df[feature_columns].copy()
y = window_df[TARGET].copy()
groups = window_df["session_id"].copy()

print("Numeric predictors:", numeric_features)
print("Categorical predictors:", categorical_features)
print(f"Total initial predictors: {len(feature_columns)}")
print(f"Window-level samples: {len(X):,}")
print(f"Independent session groups: {groups.nunique():,}")

assert y.notna().all(), "The target contains missing values."
assert np.isfinite(y.to_numpy(dtype=float)).all(), "The target contains non-finite values." 

## Step 6 — Create training, validation, and test sets

We use a two-stage `GroupShuffleSplit` based on `session_id` (the combination of `user` and `walk_id`). This keeps every temporal window from a walking session in exactly one partition. The target proportions are approximately 70% training, 15% validation, and 15% test **by session groups**; the row proportions can differ because sessions contain different numbers of windows.

The test set is held back until after model selection. Do not use test metrics to choose models or tune parameters.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# First hold out approximately 15% of session groups for the final test set.
test_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=RANDOM_STATE
)
trainval_idx, test_idx = next(test_splitter.split(X, y, groups=groups))

X_trainval = X.iloc[trainval_idx].copy()
y_trainval = y.iloc[trainval_idx].copy()
groups_trainval = groups.iloc[trainval_idx].copy()

X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# From the remaining 85% of session groups, hold out 15/85 for validation.
validation_fraction_of_remaining = 0.15 / 0.85
validation_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=validation_fraction_of_remaining,
    random_state=RANDOM_STATE + 1
)
train_idx_relative, val_idx_relative = next(
    validation_splitter.split(
        X_trainval, y_trainval, groups=groups_trainval
    )
)

X_train = X_trainval.iloc[train_idx_relative].copy()
y_train = y_trainval.iloc[train_idx_relative].copy()
groups_train = groups_trainval.iloc[train_idx_relative].copy()

X_val = X_trainval.iloc[val_idx_relative].copy()
y_val = y_trainval.iloc[val_idx_relative].copy()
groups_val = groups_trainval.iloc[val_idx_relative].copy()

# Verify no session overlaps between any partitions.
train_sessions = set(groups_train.unique())
val_sessions = set(groups_val.unique())
test_sessions = set(groups_test.unique())

assert train_sessions.isdisjoint(val_sessions), "Session leakage between train and validation."
assert train_sessions.isdisjoint(test_sessions), "Session leakage between train and test."
assert val_sessions.isdisjoint(test_sessions), "Session leakage between validation and test."

split_summary = pd.DataFrame({
    "partition": ["Train", "Validation", "Test"],
    "windows": [len(X_train), len(X_val), len(X_test)],
    "sessions": [
        groups_train.nunique(),
        groups_val.nunique(),
        groups_test.nunique()
    ],
    "window_percent": [
        100 * len(X_train) / len(X),
        100 * len(X_val) / len(X),
        100 * len(X_test) / len(X)
    ]
})
display(split_summary.round(2))

print("Session overlap checks passed: 0 overlap across all partitions.")
print("\nSpeed range by partition:")
display(pd.DataFrame({
    "train": y_train.describe(),
    "validation": y_val.describe(),
    "test": y_test.describe()
}).round(4))

## Step 7 — Fit preprocessing on training data only

The preprocessing pipeline imputes numeric features with the training median, standardizes numeric features, imputes categorical values with the most frequent training category, and one-hot encodes categories. It is fitted only on `X_train`; validation and test data are transformed using that fitted preprocessor.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
# Transform test for validation checks only; do not use test metrics yet.
X_test_processed = preprocessor.transform(X_test)

print("Training matrix:", X_train_processed.shape)
print("Validation matrix:", X_val_processed.shape)
print("Test matrix:", X_test_processed.shape)
print("Preprocessing was fitted on training data only.")

## Step 8 — Validate and visualize the processed features

Check that the transformed matrices contain finite values and have matching feature counts. The plots below are diagnostic only; they do not use validation/test results to fit preprocessing.

In [ ]:
# Convert to dense arrays for validation and plotting.
# This dataset has a small engineered feature set, so this is manageable.
def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)

train_dense = to_dense(X_train_processed)
val_dense = to_dense(X_val_processed)
test_dense = to_dense(X_test_processed)

assert np.isfinite(train_dense).all(), "Non-finite values in training features."
assert np.isfinite(val_dense).all(), "Non-finite values in validation features."
assert np.isfinite(test_dense).all(), "Non-finite values in test features."
assert train_dense.shape[1] == val_dense.shape[1] == test_dense.shape[1]

print("All transformed values are finite.")
print("Feature counts match across all partitions:", train_dense.shape[1])

numeric_train_scaled = preprocessor.named_transformers_["numeric"].transform(
    X_train[numeric_features]
)
numeric_train_df = pd.DataFrame(
    numeric_train_scaled,
    columns=numeric_features,
    index=X_train.index
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

X_train[["rssi_mean", "rssi_median", "rssi_std", "rssi_range"]].plot.box(
    ax=axes[0], showfliers=False
)
axes[0].set_title("Aggregated RSSI Features Before Scaling")
axes[0].set_ylabel("Original feature value")
axes[0].tick_params(axis="x", rotation=25)

numeric_train_df[numeric_features].plot.box(
    ax=axes[1], showfliers=False
)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Numeric Features After Scaling")
axes[1].set_ylabel("Standardized value")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Step 9 — Train Linear Regression

Fit a basic linear model on the training partition only. We will evaluate it on validation data in the comparison section. The test set remains untouched for model selection.

In [ ]:
from sklearn.linear_model import LinearRegression

linear_regression = LinearRegression()
linear_regression.fit(X_train_processed, y_train)

linear_val_predictions = linear_regression.predict(X_val_processed)
print("Linear Regression training complete.")

## Step 10 — Train Random Forest Regressor

Random Forest can learn nonlinear relationships and interactions between the engineered BLE features. Scaling is not required for tree models, but using the same preprocessed feature matrices keeps the model comparison consistent. The initial settings below are a baseline, not tuned hyperparameters.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_forest = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
random_forest.fit(X_train_processed, y_train)

random_forest_val_predictions = random_forest.predict(X_val_processed)
print("Random Forest training complete.")

## Step 11 — Compare models on the validation set

Use validation MAE, RMSE, and R² to compare Linear Regression and Random Forest. Choose the preferred model using validation performance only. These values are not final test results.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Regression does not have a single standard "accuracy" score like classification.
# This optional metric reports the percentage of predictions within a chosen
# absolute error tolerance, in the target speed's units.
ACCURACY_TOLERANCE = 0.1

def regression_metrics(y_true, y_pred, tolerance=ACCURACY_TOLERANCE):
    absolute_errors = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        f"Within ±{tolerance:g} speed units (%)": 100 * np.mean(absolute_errors <= tolerance),
    }

validation_results = pd.DataFrame({
    "Linear Regression": regression_metrics(y_val, linear_val_predictions),
    "Random Forest": regression_metrics(y_val, random_forest_val_predictions),
}).T.sort_values("MAE")

display(validation_results.round(5))

best_model_name = validation_results["MAE"].idxmin()
print(f"Selected model based on lowest validation MAE: {best_model_name}")
print(
    f"Tolerance accuracy means the percentage of predictions within "
    f"±{ACCURACY_TOLERANCE:g} speed units of the true value. "
    "It is a supplementary metric, not the standard classification accuracy."
)

## Step 12 — Final test evaluation and plots for both models

After model selection based on validation MAE, refit both baseline models on the combined training + validation data and evaluate them on the held-out test set for a transparent comparison. The model selected for the primary result remains the one chosen using validation performance; test results must not be used to tune or change the models.

The final section reports MAE, RMSE, R², and the supplementary tolerance-based accuracy for **both** Linear Regression and Random Forest. It also plots actual versus predicted speed for both models.

In [ ]:
from sklearn.base import clone

# Combine training and validation partitions after selecting the preferred model.
X_trainval_final = pd.concat([X_train, X_val], axis=0)
y_trainval_final = pd.concat([y_train, y_val], axis=0)

# Fit preprocessing on train + validation only; test data remains excluded from fitting.
final_preprocessor = clone(preprocessor)
X_trainval_final_processed = final_preprocessor.fit_transform(X_trainval_final)
X_test_final_processed = final_preprocessor.transform(X_test)

# Refit both models on the combined training + validation set.
final_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

test_predictions_by_model = {}
test_metrics_by_model = {}

for model_name, model in final_models.items():
    model.fit(X_trainval_final_processed, y_trainval_final)
    predictions = model.predict(X_test_final_processed)
    test_predictions_by_model[model_name] = predictions
    test_metrics_by_model[model_name] = regression_metrics(y_test, predictions)

test_results = pd.DataFrame(test_metrics_by_model).T
print("Final held-out test metrics for both models:")
display(test_results.round(5))

print(f"\nModel selected using validation MAE: {best_model_name}")
print(
    "Use validation metrics for model selection. The test metrics above are "
    "reported for final, held-out evaluation and should not be used for tuning."
)

# Actual vs predicted plots for both models.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

min_val = min(
    float(y_test.min()),
    min(float(np.min(pred)) for pred in test_predictions_by_model.values())
)
max_val = max(
    float(y_test.max()),
    max(float(np.max(pred)) for pred in test_predictions_by_model.values())
)

for ax, (model_name, predictions) in zip(axes, test_predictions_by_model.items()):
    ax.scatter(y_test, predictions, alpha=0.35)
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    ax.set_title(f"Actual vs Predicted — {model_name}")
    ax.set_xlabel("Actual speed")
    ax.set_ylabel("Predicted speed")

plt.tight_layout()
plt.show()

## Notes and limitations

- This split holds out walking sessions, not necessarily entire users. If the intended deployment is to predict speed for completely unseen people, run a separate user-held-out evaluation.
- Because each session has one speed label, windows from a session are correlated and do not represent independent sessions. Group splitting prevents the same session from appearing across partitions.
- The 500 ms window is an initial choice. Compare 250 ms and 1,000 ms in separate experiments, keeping the group split logic consistent.
- Random Forest parameters are initial baseline settings. If tuning them, use training/validation data or group-aware cross-validation; do not tune against the test set.
- The current RSSI forward-fill occurs before the session split but is confined to the same session/device and follows time order. For a strict streaming evaluation, ensure each imputed value uses only measurements available at that prediction time.


# Step 13 — Session-level prediction experiment

The earlier model trained on 500 ms windows even though every window in a walking session has the same speed label. This can encourage predictions near the average because individual BLE windows are noisy and highly correlated.

This experiment creates **one row per walking session**, summarising the session's BLE windows, then trains only the existing model families: Linear Regression and Random Forest. It excludes user/session IDs, walk IDs, timestamps, and target-derived values from predictors. This aligns the training unit with the dataset's label structure. It may improve performance, but cannot guarantee R² > 0.90.


In [ ]:
# Validate that each session has one speed label.
session_speed_counts = window_df.groupby("session_id")[TARGET].nunique()
if (session_speed_counts > 1).any():
    raise ValueError("Some sessions have multiple speed labels; check session construction.")

source_numeric = [
    c for c in numeric_features
    if c in window_df.columns and c != TARGET
    and pd.api.types.is_numeric_dtype(window_df[c])
]
source_categorical = [c for c in categorical_features if c in window_df.columns]

# Aggregate window-level BLE features into robust session summaries.
session_X = window_df.groupby("session_id")[source_numeric].agg(
    ["mean", "std", "median", "min", "max"]
)
session_X.columns = [f"{feature}__{stat}" for feature, stat in session_X.columns]

for col in ["rssi_mean", "rssi_median", "rssi_range", "n_observations"]:
    if col in window_df.columns:
        q = window_df.groupby("session_id")[col].quantile([0.1, 0.25, 0.75, 0.9]).unstack()
        q.columns = [f"{col}__q{int(v*100)}" for v in q.columns]
        session_X = session_X.join(q, how="left")

for col in source_categorical:
    session_X[f"{col}__mode"] = window_df.groupby("session_id")[col].agg(
        lambda s: s.mode(dropna=True).iloc[0] if not s.mode(dropna=True).empty else np.nan
    )

session_X["session_window_count"] = window_df.groupby("session_id").size()
session_y = window_df.groupby("session_id")[TARGET].first().astype(float)
session_ids = session_X.index.to_series().rename("session_id")

session_X = session_X.reset_index(drop=True)
session_y = session_y.reset_index(drop=True)
session_ids = session_ids.reset_index(drop=True)

# Remove any all-missing feature columns.
session_X = session_X.dropna(axis=1, how="all")
session_numeric = [c for c in session_X.columns if pd.api.types.is_numeric_dtype(session_X[c])]
session_categorical = [c for c in session_X.columns if c not in session_numeric]

print("Independent session examples:", len(session_X))
print("Feature count:", session_X.shape[1])
print("Speed labels per session:", int(session_y.nunique()))
display(session_y.describe().to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
window_df[TARGET].hist(bins=30, ax=axes[0])
axes[0].set_title("Window-level speed distribution")
session_y.hist(bins=25, ax=axes[1])
axes[1].set_title("Session-level speed distribution")
for ax in axes:
    ax.set_xlabel("Walking speed")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


## Step 14 — Diagnose feature relationships at session level

These correlations help identify which BLE summaries carry a signal about speed. Correlation is a diagnostic, not proof that a feature will generalise.


In [ ]:
corr_s = session_X[session_numeric].corrwith(session_y).dropna()
corr_s = corr_s.sort_values(key=lambda s: s.abs(), ascending=False)
display(corr_s.head(20).rename("Correlation with speed").to_frame().round(3))

plt.figure(figsize=(10, 6))
corr_s.head(15).sort_values().plot(kind="barh")
plt.axvline(0, linestyle="--")
plt.title("Top session-level feature correlations")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()


## Step 15 — Grouped train/validation/test split and preprocessing

Each session is one row and appears in exactly one partition. Preprocessing is fitted on training sessions only. The mean baseline is included for comparison.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

splitter_test_s = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=246)
trainval_idx_s, test_idx_s = next(
    splitter_test_s.split(session_X, session_y, groups=session_ids)
)
X_trainval_s, X_test_s = session_X.iloc[trainval_idx_s], session_X.iloc[test_idx_s]
y_trainval_s, y_test_s = session_y.iloc[trainval_idx_s], session_y.iloc[test_idx_s]
g_trainval_s = session_ids.iloc[trainval_idx_s]

splitter_val_s = GroupShuffleSplit(n_splits=1, test_size=0.1875, random_state=247)
train_rel_s, val_rel_s = next(
    splitter_val_s.split(X_trainval_s, y_trainval_s, groups=g_trainval_s)
)
X_train_s, X_val_s = X_trainval_s.iloc[train_rel_s], X_trainval_s.iloc[val_rel_s]
y_train_s, y_val_s = y_trainval_s.iloc[train_rel_s], y_trainval_s.iloc[val_rel_s]
g_train_s = g_trainval_s.iloc[train_rel_s]
g_val_s = g_trainval_s.iloc[val_rel_s]

assert set(g_train_s).isdisjoint(set(g_val_s))
assert set(g_train_s).isdisjoint(set(session_ids.iloc[test_idx_s]))
assert set(g_val_s).isdisjoint(set(session_ids.iloc[test_idx_s]))
print("Sessions train / validation / test:", len(X_train_s), len(X_val_s), len(X_test_s))
print("Speed means train / validation / test:",
      round(y_train_s.mean(), 4), round(y_val_s.mean(), 4), round(y_test_s.mean(), 4))

preprocessor_s = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]), session_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), session_categorical),
], remainder="drop")

def session_metrics(y_true, y_pred, tolerance=0.1):
    yt, yp = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return {
        "MAE": mean_absolute_error(yt, yp),
        "RMSE": np.sqrt(mean_squared_error(yt, yp)),
        "R2": r2_score(yt, yp),
        f"Within ±{tolerance:g} (%)": 100 * np.mean(np.abs(yt-yp) <= tolerance),
    }


## Step 16 — Train and tune the same two model types

Linear Regression remains Linear Regression. Random Forest remains Random Forest. Random Forest settings are tuned using grouped cross-validation on training sessions only; validation MAE selects between the two model families.


In [ ]:
linear_s = Pipeline([
    ("preprocessor", preprocessor_s),
    ("model", LinearRegression()),
])
linear_s.fit(X_train_s, y_train_s)
pred_linear_val_s = linear_s.predict(X_val_s)

rf_s = Pipeline([
    ("preprocessor", preprocessor_s),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])
rf_grid_s = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_leaf": [2, 5, 10],
    "model__max_features": [0.7, 1.0],
}
rf_search_s = GridSearchCV(
    rf_s, rf_grid_s, scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=min(5, len(X_train_s))),
    n_jobs=-1, refit=True, verbose=1,
)
rf_search_s.fit(X_train_s, y_train_s, groups=g_train_s)
pred_rf_val_s = rf_search_s.predict(X_val_s)

val_results_s = pd.DataFrame({
    "Mean Baseline": session_metrics(y_val_s, np.full(len(y_val_s), y_train_s.mean())),
    "Linear Regression": session_metrics(y_val_s, pred_linear_val_s),
    "Tuned Random Forest": session_metrics(y_val_s, pred_rf_val_s),
}).T.sort_values("MAE")
print("Best Random Forest settings:", rf_search_s.best_params_)
display(val_results_s.round(5))

selected_s = min(
    ["Linear Regression", "Tuned Random Forest"],
    key=lambda name: val_results_s.loc[name, "MAE"]
)
print("Selected by validation MAE:", selected_s)


## Step 17 — Final held-out test results

The selected model is refit on training + validation sessions and evaluated once on the held-out test sessions. Test results are not used for tuning.


In [ ]:
X_trainval_final_s = pd.concat([X_train_s, X_val_s])
y_trainval_final_s = pd.concat([y_train_s, y_val_s])

final_s = clone(linear_s if selected_s == "Linear Regression" else rf_search_s.best_estimator_)
final_s.fit(X_trainval_final_s, y_trainval_final_s)
pred_test_s = final_s.predict(X_test_s)
baseline_test_s = np.full(len(y_test_s), y_trainval_final_s.mean())

test_results_s = pd.DataFrame({
    "Mean Baseline": session_metrics(y_test_s, baseline_test_s),
    selected_s: session_metrics(y_test_s, pred_test_s),
}).T
display(test_results_s.round(5))

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test_s, pred_test_s, alpha=0.75)
lo = float(min(y_test_s.min(), np.min(pred_test_s)))
hi = float(max(y_test_s.max(), np.max(pred_test_s)))
ax.plot([lo, hi], [lo, hi], linestyle="--", label="Perfect prediction")
ax.set_title(f"Session-level actual vs predicted — {selected_s}")
ax.set_xlabel("Actual session speed")
ax.set_ylabel("Predicted session speed")
ax.legend()
plt.tight_layout()
plt.show()

print("R² >= 0.90 reached:", bool(test_results_s.loc[selected_s, "R2"] >= 0.90))


## Interpretation and next action

This experiment aligns the training examples with the dataset's labels: one speed label per session, with statistical summaries of that session's BLE windows. It may reduce window-level noise. If held-out R² remains low, the available BLE signals may not contain enough information to infer speed reliably, or the label/collection protocol needs investigation. Do not use IDs, timestamps, future windows, or speed-derived features to inflate scores.

If the real deployment goal is to work for previously unseen people, conduct a separate user-held-out evaluation. That is a different and generally harder task than predicting new sessions from known users.
